<a href="https://colab.research.google.com/github/veigaeduarda/PAnaM_Webscraping_de_Jornais/blob/main/Ghana_news_e_My_Joy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Visão Geral
Este notebook foi desenvolvido para extrair artigos de notícias de websites baseados em WordPress, utilizando a API REST. Ele busca artigos com base em um conjunto predefinido de palavras-chave e um intervalo de datas. Os dados coletados incluem títulos de artigos, URLs, datas de publicação, autores, categorias e o texto completo do corpo. A saída é salva como arquivos CSV individuais para cada site e um arquivo CSV consolidado contendo todos os artigos.

#  Scraper — Daily Graphic · My Joy Online · News Ghana
Extrai matérias via **WordPress REST API** por palavras-chave e intervalo de datas.

**O que é coletado:** Título · URL · Data · Autor · Seção/Categoria · Corpo do texto

**Saída:** arquivo `.csv` por site + um CSV consolidado com todos os sites

---
>  A WP REST API retorna no máximo 100 resultados por página. O scraper pagina automaticamente até cobrir todo o período.

## 1 · Instalação de dependências

Esta seção instala as bibliotecas Python necessárias para o funcionamento do scraper. `requests` é usada para fazer requisições HTTP aos websites, `beautifulsoup4` para analisar o conteúdo HTML, `pandas` para manipulação de dados e salvamento em CSV, e `tqdm` para exibir barras de progresso durante o processo de scraping.

In [ ]:
!pip install requests beautifulsoup4 pandas tqdm --quiet

### Resultado da Célula: Instalação de Dependências

A saída `!pip install requests beautifulsoup4 pandas tqdm --quiet` confirma que as bibliotecas necessárias para o scraper (requests, beautifulsoup4, pandas, tqdm) foram instaladas com sucesso no ambiente do Colab.

## 2 · CONFIGURAÇÃO

Esta seção é para configurar as palavras-chave usadas para pesquisar artigos. Ela define várias listas (`eixo_1` a `eixo_6` e `assunto_1` a `assunto_6`), que são então combinadas para criar uma lista `KEYWORDS` abrangente. Cada elemento em `KEYWORDS` será uma combinação de termos dessas listas (por exemplo, 'policy ultraprocessed'). Essas palavras-chave são cruciais para filtrar os artigos recuperados dos websites.

In [ ]:
eixo_1 = ['policy','program','law','legislation','regulation','restriction','tax','policies','ban','MAHA',"'Make America Healthy Again'"]
eixo_2 = ['subsidies','subsidize','subidise','incentives','incentivate','policies','policy',"'Supplemental Nutrition Assistance Program'","'food stamps'","'Farm Bill'"]
eixo_3 = ['policy','policies','program','law','legislation','regulation']
eixo_4 = ['policy','policies','program','law','legislation','regulation','restriction','ban']
eixo_5 = ['policy','policies','program','law','legislation','regulation','restriction','ban',"'National School Lunch Program'","'School Breakfast Program'","'Summer meal programs'","'National School Lunch Act'","'Farm Bill'"]
eixo_6 = ['policy','policies','program','law','legislation','regulation','restriction','ban','tax',"'Farm Bill'","'Agricultural Policy'"]

assunto_1 = ['ultraprocessed',"'processed food'","'processed foods'","'packaged foods'",'sugar',
              "'added sugar'","'ultraprocessed beverages'","'sugary drinks'","'sugary beverages'",
              "'sweefoodtened drinks'","'sweetened beverages'","'soft drinks'",'soda',"'junk food'",
              "' additives'",'GRAS',"'generally recognized as safe'"]
assunto_2 = ["'healthy food'",'produce','fruit','vegetables','crop']
assunto_3 = ["'food label'","'food labelling'","'nutrition label'","'nutrition labelling'","'front-of-package label'"]
assunto_4 = ["'food ads'","'food advertising'","'junk food marketing'"]
assunto_5 = ["'school meals'","'school lunch'","'child nutrition'","'school nutritrion'"]
assunto_6 = ['pesticides','herbicides','weedkillers','insecticides']

policies_us = ["'Supplemental Nutrition Assistance Program'","'food stamps'","'Farm Bill'","'National School Lunch Program'","'School Breakfast Program'","'Summer meal programs'","'National School Lunch Act'"]

In [ ]:
KEYWORDS = []
eixo = [eixo_1,eixo_2,eixo_3,eixo_4,eixo_5, eixo_6]
assunto = [assunto_1,assunto_2,assunto_3,assunto_4,assunto_5,assunto_6]

for i in list(range(1,7)):
  lc = eixo[i-1] # Adjusted index to be 0-based
  lt = assunto[i-1] # Adjusted index to be 0-based
  for j in lc:
    for k in lt:
      # junta as palavras-chave
      query = j+" "+k
      #troca espaços por + e acentos pelo código que o site entende
      #query = query.replace(' ','+').replace('ç','%C3%A7').replace('ã','%C3%A3').replace('í','%C3%AD').replace('ú','%25C3%25BA').replace('õ','%25C3%25B5').replace("ó","%25C3%25B3").replace('á','%25C3%25A1')
      KEYWORDS.append(query)

print(KEYWORDS)
print(len(KEYWORDS))

['policy ultraprocessed', "policy 'processed food'", "policy 'processed foods'", "policy 'packaged foods'", 'policy sugar', "policy 'added sugar'", "policy 'ultraprocessed beverages'", "policy 'sugary drinks'", "policy 'sugary beverages'", "policy 'sweefoodtened drinks'", "policy 'sweetened beverages'", "policy 'soft drinks'", 'policy soda', "policy 'junk food'", "policy ' additives'", 'policy GRAS', "policy 'generally recognized as safe'", 'program ultraprocessed', "program 'processed food'", "program 'processed foods'", "program 'packaged foods'", 'program sugar', "program 'added sugar'", "program 'ultraprocessed beverages'", "program 'sugary drinks'", "program 'sugary beverages'", "program 'sweefoodtened drinks'", "program 'sweetened beverages'", "program 'soft drinks'", 'program soda', "program 'junk food'", "program ' additives'", 'program GRAS', "program 'generally recognized as safe'", 'law ultraprocessed', "law 'processed food'", "law 'processed foods'", "law 'packaged foods'",

### Resultado da Célula: Geração de Palavras-chave

Esta saída mostra a lista completa de palavras-chave que serão usadas para a busca (`KEYWORDS`) e a contagem total de combinações geradas (1008 palavras-chave neste caso). Isso confirma que as combinações de `eixo` e `assunto` foram criadas conforme o esperado.

Esta célula define o intervalo de datas para o processo de scraping e especifica os websites alvo.

*   `DATE_START` e `DATE_END` definem as datas de início e fim (inclusive) para a publicação dos artigos.
*   `SITES` é um dicionário onde as chaves são os nomes dos sites e os valores são suas URLs base, usadas para acessar a API REST do WordPress.
*   `PER_PAGE` define o número máximo de resultados a serem recuperados por requisição da API.
*   `DELAY_SEC` introduz uma pausa entre as requisições para evitar sobrecarregar o servidor.
*   `OUTPUT_DIR` especifica o diretório onde os arquivos CSV gerados serão salvos.

In [ ]:

# ──────────────────────────────────────────────
# 📅 INTERVALO DE DATAS
# ──────────────────────────────────────────────
DATE_START = "2015-01-01"   # início (inclusive)
DATE_END   = "2024-12-31"   # fim (inclusive)

# ──────────────────────────────────────────────
# 🌐 SITES ALVO
# ──────────────────────────────────────────────
SITES = {
    "Daily Graphic"  : "https://graphic.com.gh",
    "My Joy Online"  : "https://www.myjoyonline.com",
    "News Ghana"     : "https://www.newsghana.com.gh",
}

# ──────────────────────────────────────────────
# ⚙️ PARÂMETROS TÉCNICOS (não precisa mudar)
# ──────────────────────────────────────────────
PER_PAGE   = 100          # máx. permitido pela WP API
DELAY_SEC  = 1.2          # pausa entre requisições (seja gentil com o servidor)
OUTPUT_DIR = "/content/"  # pasta de saída no Colab

print("✅ Configuração carregada.")
print(f"   Sites    : {list(SITES.keys())}")
print(f"   Keywords : {KEYWORDS}")
print(f"   Período  : {DATE_START} → {DATE_END}")

✅ Configuração carregada.
   Sites    : ['Daily Graphic', 'My Joy Online', 'News Ghana']
   Keywords : ['policy ultraprocessed', "policy 'processed food'", "policy 'processed foods'", "policy 'packaged foods'", 'policy sugar', "policy 'added sugar'", "policy 'ultraprocessed beverages'", "policy 'sugary drinks'", "policy 'sugary beverages'", "policy 'sweefoodtened drinks'", "policy 'sweetened beverages'", "policy 'soft drinks'", 'policy soda', "policy 'junk food'", "policy ' additives'", 'policy GRAS', "policy 'generally recognized as safe'", 'program ultraprocessed', "program 'processed food'", "program 'processed foods'", "program 'packaged foods'", 'program sugar', "program 'added sugar'", "program 'ultraprocessed beverages'", "program 'sugary drinks'", "program 'sugary beverages'", "program 'sweefoodtened drinks'", "program 'sweetened beverages'", "program 'soft drinks'", 'program soda', "program 'junk food'", "program ' additives'", 'program GRAS', "program 'generally recognized as

### Resultado da Célula: Carregamento da Configuração

A mensagem `✅ Configuração carregada.` e os detalhes a seguir (`Sites`, `Keywords`, `Período`) indicam que todas as variáveis de configuração, como o intervalo de datas, os sites alvo e as palavras-chave, foram lidas e estão prontas para serem usadas pelo scraper.

## 3 · Funções principais

Esta seção define as funções principais usadas pelo scraper:

*   `clean_html(raw_html)`: Remove tags HTML de uma string e normaliza espaços em branco, retornando texto limpo.
*   `parse_date(date_str)`: Converte uma string de data formatada em ISO 8601 para o formato `YYYY-MM-DD`.
*   `get_categories(post, base_url, session)`: Recebe um objeto `post`, a URL base e uma sessão de requisições para resolver IDs de categoria em nomes legíveis por humanos, fazendo chamadas adicionais à API.
*   `get_author_name(post, base_url, session)`: Resolve um ID de autor de um objeto `post` em um nome de autor legível por humanos usando chamadas adicionais à API.
*   `fetch_posts_for_keyword(site_name, base_url, keyword, date_start, date_end, session)`: Esta é a função principal que lida com a busca de posts da API REST do WordPress. Ela constrói requisições da API com a palavra-chave, o intervalo de datas e os parâmetros de paginação fornecidos. Ela pagina automaticamente os resultados, lida com as respostas da API e chama outras funções auxiliares para limpar dados e resolver nomes de autor/categoria. Inclui tratamento de erros para problemas de rede e erros HTTP.

In [ ]:
import requests
import pandas as pd
import time
import os
import re
from datetime import datetime
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (academic research scraper; contact: your@email.com)"
}


def clean_html(raw_html: str) -> str:
    """Remove tags HTML e normaliza espaços em branco."""
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_date(date_str: str) -> str:
    """Normaliza data ISO 8601 para YYYY-MM-DD."""
    try:
        return datetime.fromisoformat(date_str.replace("Z", "+00:00")).strftime("%Y-%m-%d")
    except Exception:
        return date_str


def get_categories(post: dict, base_url: str, session: requests.Session) -> str:
    """Resolve os IDs de categoria para nomes legíveis."""
    cat_ids = post.get("categories", [])
    if not cat_ids:
        return ""
    names = []
    for cid in cat_ids:
        try:
            r = session.get(
                f"{base_url}/wp-json/wp/v2/categories/{cid}",
                headers=HEADERS, timeout=10
            )
            if r.ok:
                names.append(r.json().get("name", str(cid)))
        except Exception:
            names.append(str(cid))
        time.sleep(0.3)
    return " | ".join(names)


def get_author_name(post: dict, base_url: str, session: requests.Session) -> str:
    """Resolve o ID de autor para nome legível."""
    author_id = post.get("author")
    if not author_id:
        return ""
    try:
        r = session.get(
            f"{base_url}/wp-json/wp/v2/users/{author_id}",
            headers=HEADERS, timeout=10
        )
        if r.ok:
            return r.json().get("name", str(author_id))
    except Exception:
        pass
    return str(author_id)


def fetch_posts_for_keyword(
    site_name: str,
    base_url: str,
    keyword: str,
    date_start: str,
    date_end: str,
    session: requests.Session,
) -> list[dict]:
    """
    Busca todos os posts da WP REST API para uma palavra-chave
    no intervalo de datas. Pagina automaticamente.
    """
    endpoint = f"{base_url}/wp-json/wp/v2/posts"
    # Caches de autor e categoria para evitar requisições repetidas
    author_cache = {}
    cat_cache = {}
    results = []
    page = 1

    while True:
        params = {
            "search"   : keyword,
            "after"    : f"{date_start}T00:00:00",
            "before"   : f"{date_end}T23:59:59",
            "per_page" : PER_PAGE,
            "page"     : page,
            "_fields"  : "id,date,link,title,content,excerpt,author,categories",
            "status"   : "publish",
            "orderby"  : "date",
            "order"    : "asc",
        }

        try:
            r = session.get(endpoint, params=params, headers=HEADERS, timeout=20)
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  Erro de rede [{site_name} · '{keyword}' · pág.{page}]: {e}")
            break

        if r.status_code == 400:
            # Página além do total — fim da paginação
            break
        if not r.ok:
            print(f"  ⚠️  HTTP {r.status_code} [{site_name} · '{keyword}' · pág.{page}]")
            break

        posts = r.json()
        if not posts:
            break

        total_pages = int(r.headers.get("X-WP-TotalPages", 1))

        for post in posts:
            # ── Autor ──────────────────────────────────────────────
            author_id = post.get("author")
            if author_id not in author_cache:
                author_cache[author_id] = get_author_name(post, base_url, session)
            author_name = author_cache[author_id]

            # ── Categorias ─────────────────────────────────────────
            cat_key = tuple(post.get("categories", []))
            if cat_key not in cat_cache:
                cat_cache[cat_key] = get_categories(post, base_url, session)
            categories = cat_cache[cat_key]

            results.append({
                "site"      : site_name,
                "keyword"   : keyword,
                "date"      : parse_date(post.get("date", "")),
                "title"     : clean_html(post["title"]["rendered"]),
                "url"       : post.get("link", ""),
                "author"    : author_name,
                "categories": categories,
                "body"      : clean_html(post["content"]["rendered"]),
                "excerpt"   : clean_html(post["excerpt"]["rendered"]),
                "post_id"   : post.get("id"),
            })

        if page >= total_pages:
            break

        page += 1
        time.sleep(DELAY_SEC)

    return results


print("✅ Funções carregadas.")

✅ Funções carregadas.


### Resultado da Célula: Funções do Scraper

A saída `✅ Funções carregadas.` simplesmente confirma que todas as funções auxiliares e a função principal de busca (`fetch_posts_for_keyword`) foram definidas e estão prontas para serem chamadas pelo código principal do scraper.

## 4 · Execução do scraper

Esta célula orquestra todo o processo de scraping.

1.  Ela inicializa uma lista vazia `all_records` para armazenar todos os artigos coletados.
2.  Ela itera sobre cada `site` definido no dicionário `SITES`.
3.  Para cada site, ela então itera sobre cada `keyword` na lista `KEYWORDS`.
4.  Ela chama `fetch_posts_for_keyword` para recuperar artigos para o site e a palavra-chave atuais.
5.  Após buscar artigos para todas as palavras-chave em um determinado site, ela remove duplicatas de artigos por suas URLs para evitar entradas redundantes.
6.  Ela salva os artigos coletados para aquele site específico em um arquivo CSV separado.
7.  Finalmente, ela estende `all_records` com os artigos únicos do site atual.

In [ ]:
all_records = []

with requests.Session() as session:
    for site_name, base_url in SITES.items():
        print(f"\n{'═'*55}")
        print(f" {site_name} ({base_url})")
        print(f"{'═'*55}")

        site_records = []

        for keyword in tqdm(KEYWORDS, desc=f"{site_name}", unit="kw"):
            print(f"   Buscando: '{keyword}'...")
            records = fetch_posts_for_keyword(
                site_name, base_url, keyword,
                DATE_START, DATE_END, session
            )
            print(f"     → {len(records)} matérias encontradas")
            site_records.extend(records)
            time.sleep(DELAY_SEC)

        # Deduplica por URL (mesmo artigo pode aparecer em múltiplas buscas)
        df_site = pd.DataFrame(site_records)
        if not df_site.empty:
            before = len(df_site)
            df_site = df_site.drop_duplicates(subset="url")
            after = len(df_site)
            if before != after:
                print(f"   {before - after} duplicatas removidas")

            # Salva CSV individual por site
            safe_name = site_name.lower().replace(" ", "_")
            csv_path = os.path.join(OUTPUT_DIR, f"{safe_name}_{DATE_START[:4]}_{DATE_END[:4]}.csv")
            df_site.to_csv(csv_path, index=False, encoding="utf-8-sig")
            print(f"   Salvo: {csv_path} ({len(df_site)} artigos)")

        all_records.extend(df_site.to_dict("records") if not df_site.empty else [])

print(f"\n{'═'*55}")
print(f"Coleta concluída! Total bruto: {len(all_records)} artigos")


═══════════════════════════════════════════════════════
 Daily Graphic (https://graphic.com.gh)
═══════════════════════════════════════════════════════


Daily Graphic:   0%|          | 0/387 [00:00<?, ?kw/s]

   Buscando: 'policy ultraprocessed'...
  ⚠️  HTTP 403 [Daily Graphic · 'policy ultraprocessed' · pág.1]
     → 0 matérias encontradas
   Buscando: 'policy 'processed food''...
  ⚠️  HTTP 403 [Daily Graphic · 'policy 'processed food'' · pág.1]
     → 0 matérias encontradas
   Buscando: 'policy 'processed foods''...
  ⚠️  HTTP 403 [Daily Graphic · 'policy 'processed foods'' · pág.1]
     → 0 matérias encontradas
   Buscando: 'policy 'packaged foods''...
  ⚠️  HTTP 403 [Daily Graphic · 'policy 'packaged foods'' · pág.1]
     → 0 matérias encontradas
   Buscando: 'policy sugar'...
  ⚠️  HTTP 403 [Daily Graphic · 'policy sugar' · pág.1]
     → 0 matérias encontradas
   Buscando: 'policy 'added sugar''...
  ⚠️  HTTP 403 [Daily Graphic · 'policy 'added sugar'' · pág.1]
     → 0 matérias encontradas
   Buscando: 'policy 'ultraprocessed beverages''...
  ⚠️  HTTP 403 [Daily Graphic · 'policy 'ultraprocessed beverages'' · pág.1]
     → 0 matérias encontradas
   Buscando: 'policy 'sugary drinks''

My Joy Online:   0%|          | 0/387 [00:00<?, ?kw/s]

   Buscando: 'policy ultraprocessed'...
     → 0 matérias encontradas
   Buscando: 'policy 'processed food''...
     → 102 matérias encontradas
   Buscando: 'policy 'processed foods''...
     → 41 matérias encontradas
   Buscando: 'policy 'packaged foods''...
     → 12 matérias encontradas
   Buscando: 'policy sugar'...
     → 228 matérias encontradas
   Buscando: 'policy 'added sugar''...
     → 83 matérias encontradas
   Buscando: 'policy 'ultraprocessed beverages''...
     → 0 matérias encontradas
   Buscando: 'policy 'sugary drinks''...
     → 8 matérias encontradas
   Buscando: 'policy 'sugary beverages''...
     → 4 matérias encontradas
   Buscando: 'policy 'sweefoodtened drinks''...
     → 0 matérias encontradas
   Buscando: 'policy 'sweetened beverages''...
     → 17 matérias encontradas
   Buscando: 'policy 'soft drinks''...
     → 30 matérias encontradas
   Buscando: 'policy soda'...
     → 19 matérias encontradas
   Buscando: 'policy 'junk food''...
     → 21 matérias encont

News Ghana:   0%|          | 0/387 [00:00<?, ?kw/s]

   Buscando: 'policy ultraprocessed'...
     → 0 matérias encontradas
   Buscando: 'policy 'processed food''...
     → 109 matérias encontradas
   Buscando: 'policy 'processed foods''...
     → 51 matérias encontradas
   Buscando: 'policy 'packaged foods''...
     → 11 matérias encontradas
   Buscando: 'policy sugar'...
     → 222 matérias encontradas
   Buscando: 'policy 'added sugar''...
     → 61 matérias encontradas
   Buscando: 'policy 'ultraprocessed beverages''...
     → 0 matérias encontradas
   Buscando: 'policy 'sugary drinks''...
     → 8 matérias encontradas
   Buscando: 'policy 'sugary beverages''...
     → 6 matérias encontradas
   Buscando: 'policy 'sweefoodtened drinks''...
     → 0 matérias encontradas
   Buscando: 'policy 'sweetened beverages''...
     → 24 matérias encontradas
   Buscando: 'policy 'soft drinks''...
     → 25 matérias encontradas
   Buscando: 'policy soda'...
     → 13 matérias encontradas
   Buscando: 'policy 'junk food''...
     → 21 matérias encont

### Resultado da Célula: Execução do Scraper

Esta é a parte central da execução. Podemos observar:

*   **Daily Graphic:** Para este site, o scraper retorna consistentemente `⚠️ HTTP 403`. O código de status HTTP 403 significa "Forbidden" (Proibido). Isso indica que o servidor do Daily Graphic recusou o acesso à API, provavelmente devido a restrições de segurança ou porque ele detectou a atividade de scraping e a bloqueou. Consequentemente, **nenhum artigo foi coletado deste site**.
*   **My Joy Online e News Ghana:** Para estes dois sites, o scraper conseguiu encontrar e coletar artigos para a maioria das palavras-chave, indicando que a API desses sites permitiu o acesso. As mensagens `→ XX matérias encontradas` confirmam a coleta bem-sucedida.
*   **Deduplicação e Salvamento:** Após a coleta de cada site, o scraper informa quantas duplicatas foram removidas (se houver) e o caminho e a quantidade de artigos salvos no arquivo CSV individual do site.

## 5 · CSV consolidado (todos os sites)

Esta célula processa todos os artigos coletados de `all_records` para criar um único arquivo CSV consolidado.

1.  Ela converte `all_records` em um DataFrame do pandas (`df_all`).
2.  Se `df_all` não estiver vazio, ela converte a coluna 'date' em objetos datetime e ordena o DataFrame por site e data.
3.  Ela reordena as colunas para melhor legibilidade.
4.  Finalmente, ela salva o DataFrame `df_all` em um arquivo CSV consolidado chamado `africa_news_consolidated_YYYY_YYYY.csv` no `OUTPUT_DIR` especificado. Também imprime estatísticas de resumo como o número total de artigos únicos, o intervalo de datas coberto e a contagem de artigos por site e por palavra-chave.

In [ ]:
df_all = pd.DataFrame(all_records)

if df_all.empty:
    print("  Nenhum resultado encontrado. Verifique as palavras-chave e o período.")
else:
    # Ordena por site e data
    df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
    df_all = df_all.sort_values(["site", "date"]).reset_index(drop=True)

    # Reordena colunas para leitura mais clara
    cols = ["site", "keyword", "date", "title", "author", "categories", "url", "excerpt", "body", "post_id"]
    df_all = df_all[[c for c in cols if c in df_all.columns]]

    # Salva CSV consolidado
    consolidated_path = os.path.join(OUTPUT_DIR, f"africa_news_consolidated_{DATE_START[:4]}_{DATE_END[:4]}.csv")
    df_all.to_csv(consolidated_path, index=False, encoding="utf-8-sig")

    print(f"CSV consolidado salvo em: {consolidated_path}")
    print(f"   Total de artigos únicos : {len(df_all)}")
    print(f"   Período coberto         : {df_all['date'].min().date()} → {df_all['date'].max().date()}")
    print("\n Artigos por site:")
    print(df_all.groupby("site").size().to_string())
    print("\n Artigos por palavra-chave:")
    print(df_all.groupby("keyword").size().to_string())

CSV consolidado salvo em: /content/africa_news_consolidated_2015_2024.csv
   Total de artigos únicos : 26763
   Período coberto         : 2015-01-01 → 2024-12-31

 Artigos por site:
site
My Joy Online    13304
News Ghana       13459

 Artigos por palavra-chave:
keyword
'Farm Bill' 'child nutrition'                           7
'Farm Bill' 'healthy food'                             83
'Farm Bill' crop                                      300
'Farm Bill' fruit                                      81
'Farm Bill' insecticides                                1
'Farm Bill' pesticides                                  1
'Farm Bill' produce                                   881
'Farm Bill' vegetables                                 30
'Make America Healthy Again' 'packaged foods'           1
'Make America Healthy Again' 'processed food'           3
'Make America Healthy Again' sugar                      6
'National School Lunch Act' 'school lunch'             23
'National School Lunch Act' 'schoo

### Resultado da Célula: CSV Consolidado

Esta saída mostra o resultado final do processamento:

*   Confirma que o arquivo CSV consolidado foi salvo, incluindo o caminho.
*   Detalha o `Total de artigos únicos` (26763). Este número corresponde à soma dos artigos coletados de "My Joy Online" e "News Ghana", pois o "Daily Graphic" não retornou nenhum.
*   Apresenta o `Período coberto` pela coleta.
*   A seção `Artigos por site` confirma explicitamente que apenas "My Joy Online" e "News Ghana" contribuíram com artigos, cada um com mais de 13.000 entradas. O "Daily Graphic" não aparece nesta lista, reforçando que não houve sucesso na coleta.
*   A lista `Artigos por palavra-chave` mostra a distribuição de artigos encontrados por cada palavra-chave nos sites que permitiram a extração.

## 6 · Download dos arquivos gerados

Esta célula final oferece uma maneira de baixar todos os arquivos CSV gerados diretamente do Google Colab.

1.  Ela usa `glob.glob` para encontrar todos os arquivos CSV no `OUTPUT_DIR`.
2.  Ela lista os arquivos encontrados juntamente com seus tamanhos.
3.  Em seguida, ela usa `google.colab.files.download()` para iniciar o download de cada arquivo CSV para sua máquina local.

In [ ]:
from google.colab import files
import glob

csvs = glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))
print(f" {len(csvs)} arquivo(s) disponível(is) para download:")
for f in csvs:
    size_kb = os.path.getsize(f) / 1024
    print(f"   {os.path.basename(f):50s} {size_kb:8.1f} KB")

print("\n  Iniciando download...")
for f in csvs:
    files.download(f)

 3 arquivo(s) disponível(is) para download:
   africa_news_consolidated_2015_2024.csv             151206.5 KB
   my_joy_online_2015_2024.csv                         78172.0 KB
   news_ghana_2015_2024.csv                            73034.6 KB

  Iniciando download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Resultado da Célula: Download dos Arquivos Gerados

A saída `X arquivo(s) disponível(is) para download:` lista os arquivos CSV gerados (`africa_news_consolidated`, `my_joy_online`, `news_ghana`) e seus respectivos tamanhos em KB. Em seguida, a mensagem `Iniciando download...` indica que o processo de download desses arquivos para sua máquina local foi iniciado, permitindo que você acesse os dados coletados.

### Por que apenas dois jornais retornaram resultados?

Como observado na **Execução do Scraper**, o site "Daily Graphic" retornou repetidamente o erro **`HTTP 403`**. Este código de status significa "Forbidden" (Proibido), indicando que o servidor do "Daily Graphic" recusou as requisições de acesso do scraper. Isso pode ocorrer por várias razões, como:

*   **Bloqueio de IP:** O site pode ter detectado um padrão de acesso automatizado e bloqueado o IP do Colab.
*   **Restrições de API:** A API do WordPress do site pode ter configurações que impedem o acesso externo ou exigem autenticação específica não fornecida.
*   **Firewall:** Um firewall no servidor pode estar bloqueando as requisições.

Devido a essa recusa de acesso, não foi possível coletar nenhum artigo do "Daily Graphic", resultando em dados apenas de "My Joy Online" e "News Ghana", cujas APIs permitiram a busca.

---
### Aviso Final

Este código foi comentado por uma inteligência artificial para facilitar a compreensão. As explicações foram revisadas e validadas por uma especialista humana.